# hERG Docking — Exploratory Bonus

**Separate from the main project.** This notebook is an *exploratory computational
docking* experiment, kept distinct from `CardioSafeAI_MM.ipynb` (the predictive
pipeline) and `hERG_Structure_Drugs.ipynb` (the structural illustration). Nothing
here feeds the poster's reported AUROC / APD90 numbers.

| | This notebook | The other two notebooks |
|---|---|---|
| Question | "*Where* might these drugs sit in the pocket?" | "*Whether* and *how much* they block, and what the AP looks like" |
| Method | AutoDock Vina (docking — a computational prediction of pose) | Trained chemistry models + O'Hara-Rudy electrophysiology |
| Status | Bonus / stretch goal — pose hypotheses only | Core results, validated on TDC benchmark |
| What it is *not* | An experimentally measured structure; not a substitute for cryo-EM | — |

> ⚠️ **Docking is a prediction, not a measurement.** Scores and poses produced
> here are hypotheses, not affinities. We will explicitly validate the pipeline
> by re-docking astemizole into PDB 8ZYO and checking whether the predicted
> pose recovers the published cryo-EM pose before trusting any pose for our
> own compounds.


## Plan

1. **Validate the pipeline (sanity check).**
   Re-dock astemizole against the experimentally astemizole-bound hERG structure
   (PDB **8ZYO**, the same structure shown in Part 4 of `hERG_Structure_Drugs.ipynb`).
   Success criterion: the top-ranked Vina pose has heavy-atom RMSD ≲ 2 Å vs the
   cryo-EM ligand position. If we can't recover the known pose, we can't trust
   any docked pose for our own compounds either, and we report that honestly.

2. **Dock the four poster compounds** *(excluding astemizole itself, which we
   used as the positive control above)*:
   - Cisapride
   - Quinidine
   - Terfenadine
   - Rimantadine *(negative control — predicted non-blocker; should *not* yield
     a good pocket fit, or should score weakly)*
   We also include Oleandomycin only if the docking box can accommodate a
   macrolide of that size (likely too large for the cavity — that itself is
   informative).

3. **Report** the Vina score for each compound, the top pose against the same
   cryo-EM cavity, and a one-paragraph honest discussion of what docking
   scores *can* and *cannot* tell us about clinical blockade.

Before writing any docking code, the next step is to verify the toolchain is
actually installable on this machine. That report is generated outside the
notebook and pasted here once the install path is decided.


## Stage 2 — Install toolchain and prepare inputs

Toolchain (chosen in the feasibility report): **AutoDock Vina v1.2.7 prebuilt
macOS-arm64 binary** (called via `subprocess`) + **meeko** (ligand prep) +
**openbabel-wheel** (receptor prep). The Vina Python bindings are *not* used —
no macOS-arm64 wheel exists for them on PyPI and a from-source build against
Boost is the rabbit-hole we explicitly chose to avoid.

This stage:
1. Installs the toolchain (idempotent — cells check first and skip if already there).
2. Splits the experimental hERG-astemizole structure (PDB 8ZYO) into a
   **protein-only receptor** + an extracted **astemizole ground-truth pose**.
3. Converts the protein to **PDBQT** (Vina's required format).
4. Defines the **docking box** by centring on the experimental astemizole position,
   so docking is constrained to the real Y652 / F656 cavity.

> ⚠️  Still no docking happens in this stage. We only set up inputs and verify
> the box covers the cavity.


In [ ]:
# === Toolchain check (and install if missing) ===
# Idempotent: re-running this cell is safe.
import subprocess, shutil, sys, os
from pathlib import Path

PROJECT = Path("/Users/prateekmethwani/Documents/ai-drug-project")
os.chdir(PROJECT)
VINA_BIN = PROJECT / "tools" / "vina"

def sh(cmd, check=True):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, check=check)

if not VINA_BIN.exists():
    print("downloading vina v1.2.7 macOS arm64 binary...")
    (PROJECT / "tools").mkdir(exist_ok=True)
    sh(f"curl -fsSL -o {VINA_BIN} "
       f"https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_mac_aarch64")
    sh(f"chmod +x {VINA_BIN}")
    sh(f"xattr -d com.apple.quarantine {VINA_BIN} 2>/dev/null || true", check=False)

ver = sh(f"{VINA_BIN} --version").stdout.strip()
print(f"vina binary    : {VINA_BIN}  →  {ver}")

# Python deps
try:
    import meeko, gemmi
    from openbabel import openbabel as ob
except ImportError as e:
    print(f"missing python deps ({e}); installing...")
    sh(f"{sys.executable} -m pip install -q meeko openbabel-wheel gemmi")
    import meeko, gemmi
    from openbabel import openbabel as ob

print(f"meeko          : {meeko.__version__}")
print(f"openbabel      : {ob.OBReleaseVersion()}")
print(f"gemmi          : {gemmi.__version__}")


### 2.1 — Split the cryo-EM structure into receptor + ground-truth ligand

PDB **8ZYO** contains both the hERG channel (ATOM records, 6,170 lines) and the
co-resolved astemizole molecule (HETATM records, residue name `XB7`, 68 atoms
across two alternate conformers A/B with occupancies 0.30/0.70). For docking we
need them in two separate files:

- The **receptor** = protein only, no ligand, no waters.
- The **ground-truth ligand** = the astemizole atoms as resolved in the cryo-EM
  density. We'll need these for RMSD validation later (re-dock astemizole into
  this same site and check we recover its experimental pose).

We keep the **B conformer** (higher occupancy, 0.70) as the canonical ground-truth.


In [ ]:
# === Split 8ZYO into protein receptor + astemizole ground-truth ===
import numpy as np
from pathlib import Path

SRC      = Path("data/structures/8ZYO.pdb")
DOCK_DIR = Path("data/docking"); DOCK_DIR.mkdir(parents=True, exist_ok=True)

protein_lines, xb7_all_lines, xb7_B_lines = [], [], []
xb7_B_coords = []
for line in SRC.read_text().splitlines():
    if line.startswith(("ATOM  ", "TER")):
        protein_lines.append(line)
    elif line.startswith("HETATM") and line[17:20].strip() == "XB7":
        xb7_all_lines.append(line)
        alt = line[16]                       # alt-loc indicator
        if alt in (" ", "B"):                # B = higher-occupancy conformer
            xb7_B_lines.append(line)
            xb7_B_coords.append((float(line[30:38]),
                                 float(line[38:46]),
                                 float(line[46:54])))

protein_pdb = DOCK_DIR / "8ZYO_protein.pdb"
xb7_truth   = DOCK_DIR / "8ZYO_XB7_ground_truth.pdb"
xb7_all     = DOCK_DIR / "8ZYO_XB7_all_altlocs.pdb"

protein_pdb.write_text(
    "REMARK   Protein chains only from 8ZYO (no HETATMs)\n"
    + "\n".join(protein_lines) + "\nEND\n")
xb7_truth.write_text(
    "REMARK   Astemizole (XB7) ground-truth pose — B alt-loc, occupancy 0.70\n"
    + "\n".join(xb7_B_lines) + "\nEND\n")
xb7_all.write_text(
    "REMARK   Astemizole (XB7) — both alt-locs A and B\n"
    + "\n".join(xb7_all_lines) + "\nEND\n")

print(f"protein_pdb       : {protein_pdb}  ({protein_pdb.stat().st_size:,} bytes, "
      f"{len(protein_lines):,} ATOM/TER lines)")
print(f"ground-truth XB7  : {xb7_truth}  ({len(xb7_B_lines)} atoms, B conformer)")
print(f"both alt-locs     : {xb7_all}  ({len(xb7_all_lines)} atoms)")


### 2.2 — Convert receptor to PDBQT (Vina's input format)

PDBQT = PDB + AutoDock atom types (`A`, `OA`, `NA`, …) + partial charges. Vina
needs this for the receptor.

We use **`obabel`** (from the just-installed `openbabel-wheel`) rather than
meeko's `mk_prepare_receptor.py`. The reason: meeko's polymer prep insists on
adding/reconciling hydrogens against a residue-template library, and the
cryo-EM structure has no hydrogens — that causes meeko 0.7.1 to error with
`RuntimeError: Updated 1 H positions but deleted 13`. `obabel`'s rigid-receptor
mode (`-xr`) handles a hydrogenless cryo-EM PDB directly. Partial charges show
as `0.000` in the PDBQT — that's fine: **Vina's default scoring function is
atom-type-based, not charge-based** (charges only matter if you switch to the
AD4 scoring function).


In [ ]:
# === Convert protein PDB → PDBQT with openbabel ===
import subprocess
from pathlib import Path

receptor_pdbqt = Path("data/docking/8ZYO_receptor.pdbqt")

r = subprocess.run(
    [".venv/bin/obabel", "data/docking/8ZYO_protein.pdb",
     "-O", str(receptor_pdbqt), "-xr"],
    capture_output=True, text=True,
)
# obabel prints "1 molecule converted" to stderr; treat non-zero return as fatal
if r.returncode != 0:
    raise RuntimeError(f"obabel failed:\n{r.stderr}")

# Sanity-check the result
text = receptor_pdbqt.read_text()
atom_lines = [l for l in text.splitlines() if l.startswith("ATOM")]
atom_types = {}
for l in atom_lines:
    t = l.split()[-1]
    atom_types[t] = atom_types.get(t, 0) + 1

print(f"receptor_pdbqt    : {receptor_pdbqt}  ({receptor_pdbqt.stat().st_size:,} bytes)")
print(f"  atom count      : {len(atom_lines):,}")
print(f"  atom-type mix   : {dict(sorted(atom_types.items(), key=lambda kv: -kv[1]))}")
print(f"  → expected AD types present: C (sp3), A (aromatic), OA (O acceptor), NA (N acceptor)")


### 2.3 — Define the docking box

We centre the search box on the **experimentally observed astemizole position**.
That hard-constrains docking to the real Y652 / F656 cavity, which is both the
biologically correct site and an honest controlled experiment: we're testing
whether Vina can re-find a pose in a cavity it's already pointed at.

Box size = 24 Å per side. The astemizole molecule's longest dimension in the
cavity is ~15 Å, so 24 Å gives ~4 Å of slack on every side — enough for our
own poster compounds (Cisapride, Terfenadine etc., similar size) without
wasting compute on a giant box.


In [ ]:
# === Compute the docking box ===
import numpy as np
from pathlib import Path

xb7_truth = Path("data/docking/8ZYO_XB7_ground_truth.pdb")

coords = []
for line in xb7_truth.read_text().splitlines():
    if line.startswith("HETATM"):
        coords.append((float(line[30:38]), float(line[38:46]), float(line[46:54])))
coords = np.array(coords)

center = coords.mean(axis=0)
extent = coords.max(axis=0) - coords.min(axis=0)

BOX_SIZE = 24.0   # Å per side; tunable

# Save the box parameters to a small text file so other cells (and any
# subprocess vina call) can reuse them without re-deriving.
box_path = Path("data/docking/box_params.txt")
box_path.write_text(
    f"center_x {center[0]:.3f}\n"
    f"center_y {center[1]:.3f}\n"
    f"center_z {center[2]:.3f}\n"
    f"size_x   {BOX_SIZE}\n"
    f"size_y   {BOX_SIZE}\n"
    f"size_z   {BOX_SIZE}\n"
)

print(f"box centre (Å)    : ({center[0]:7.2f}, {center[1]:7.2f}, {center[2]:7.2f})")
print(f"box size   (Å)    : ({BOX_SIZE}, {BOX_SIZE}, {BOX_SIZE})")
print(f"ligand extent (Å) : ({extent[0]:5.1f}, {extent[1]:5.1f}, {extent[2]:5.1f})  "
      f"→ slack ≈ {(BOX_SIZE - extent.max())/2:.1f} Å each side")
print(f"box params saved  : {box_path}")


## Stage 3 — Validation: re-docking astemizole (positive control)

Before we trust any docked pose for our four poster compounds, we run a
controlled sanity check: dock **astemizole** back into the *same* hERG receptor
we have the experimental complex for, and see if Vina recovers the cryo-EM pose.

The key rule of this experiment: **the docked ligand is built from SMILES** (a
fresh 3D conformer), *not* from the crystal coordinates. If we started from the
experimental atoms, the test would be rigged — Vina would just need to wiggle a
few Å. By starting from a clean RDKit conformer (random ETKDG embed → MMFF
minimised), we force Vina to find the pose *independently*, just as it will
have to for our unknown compounds.

**Success criterion:** heavy-atom RMSD of the top-ranked Vina pose vs the
cryo-EM XB7 atoms ≲ 2 Å. This is the conventional threshold for "the docking
recovered the experimental answer" (Trott & Olson 2010 used ≤ 2 Å as their
benchmark). If we miss this, we don't trust the setup on the 4 unknowns.


In [ ]:
# === 1) Prepare astemizole as a docking ligand ===
# Canonical SMILES taken from the RCSB Chemical Component Dictionary for XB7
# (https://data.rcsb.org/rest/v1/core/chemcomp/XB7). InChI Key confirms
# identity: GXDALQBWZGODGZ-UHFFFAOYSA-N = astemizole.
#
# Build 3D with RDKit (ETKDGv3, seeded for reproducibility), minimise with MMFF,
# then convert to PDBQT via meeko's mk_prepare_ligand.py CLI.
import subprocess
from rdkit import Chem
from rdkit.Chem import AllChem

ASTEMIZOLE_SMILES = "COc1ccc(cc1)CCN2CCC(CC2)Nc3nc4ccccc4n3Cc5ccc(cc5)F"

mol = Chem.MolFromSmiles(ASTEMIZOLE_SMILES)
assert mol.GetNumHeavyAtoms() == 34, "expected 34 heavy atoms (C28FN4O)"
mol_h = Chem.AddHs(mol)
params = AllChem.ETKDGv3(); params.randomSeed = 42
AllChem.EmbedMolecule(mol_h, params)
AllChem.MMFFOptimizeMolecule(mol_h, maxIters=500)

sdf_path   = Path("data/docking/astemizole_query.sdf")
pdbqt_path = Path("data/docking/astemizole_query.pdbqt")

w = Chem.SDWriter(str(sdf_path)); w.write(mol_h); w.close()
r = subprocess.run([".venv/bin/mk_prepare_ligand.py",
                    "-i", str(sdf_path), "-o", str(pdbqt_path)],
                   capture_output=True, text=True, check=True)
torsdof = next((l for l in pdbqt_path.read_text().splitlines() if l.startswith("TORSDOF")),
               "(none)")

print(f"ligand SDF  : {sdf_path}  ({sdf_path.stat().st_size:,} B)")
print(f"ligand PDBQT: {pdbqt_path}  ({pdbqt_path.stat().st_size:,} B)")
print(f"{torsdof}  ← rotatable bonds Vina will search")


### Run AutoDock Vina

Receptor, ligand, and box are all already prepped from Stage 2. We use:
- **exhaustiveness = 16** (Vina's default is 8; doubling gives better convergence
  without much wall-time cost on this small box)
- **num_modes = 9** (the default; keep top 9 distinct poses)
- **seed = 42** for reproducibility

The Vina CLI binary (`tools/vina`) is called via `subprocess` — we deliberately
skipped the `vina` Python module because no macOS-arm64 wheel exists for it.


In [ ]:
# === 2) Run Vina ===
import time, subprocess
from pathlib import Path

box = dict(l.split() for l in Path("data/docking/box_params.txt").read_text().splitlines())
out_pdbqt = Path("data/docking/astemizole_docked.pdbqt")

t0 = time.time()
r = subprocess.run([
    "tools/vina",
    "--receptor", "data/docking/8ZYO_receptor.pdbqt",
    "--ligand",   "data/docking/astemizole_query.pdbqt",
    "--center_x", box["center_x"], "--center_y", box["center_y"], "--center_z", box["center_z"],
    "--size_x",   box["size_x"],   "--size_y",   box["size_y"],   "--size_z",   box["size_z"],
    "--exhaustiveness", "16", "--num_modes", "9", "--seed", "42",
    "--out", str(out_pdbqt),
], capture_output=True, text=True, check=True)
print(f"Vina finished in {time.time()-t0:.1f}s")

# Print the per-mode affinity table (the bit at the end of Vina's stdout)
print("\n" + "\n".join(l for l in r.stdout.splitlines()
                        if l.startswith(("mode", "  ", "-----", "     "))))
print(f"\noutput: {out_pdbqt}  ({out_pdbqt.stat().st_size:,} B)")


### Validate by RMSD vs the experimental pose

We compute **heavy-atom RMSD, no superposition, with full chemical-symmetry
awareness** — the standard convention for docking-pose validation:

- *No superposition*: the RMSD is in absolute binding-site coordinates. A
  pose that's "the right shape but translated by 5 Å" should *fail*, because
  Vina's job is to find the right *place* too.
- *Symmetry awareness*: astemizole has para-disubstituted aromatic rings that
  can be flipped 180° without changing the chemistry. We enumerate all atom-
  index automorphisms and report the minimum RMSD over them — otherwise a
  ring-flip would inflate the number.

Implementation path: openbabel converts both PDB files to SDF (so bond orders
are preserved and RDKit can parse the ligand cleanly), then RDKit enumerates
substructure-match automorphisms and we compute in-place RMSD for each.


In [ ]:
# === 3) Extract top pose and compute symmetry-aware in-place RMSD ===
import subprocess
from pathlib import Path
from rdkit import Chem

docked     = Path("data/docking/astemizole_docked.pdbqt")
truth_pdb  = Path("data/docking/8ZYO_XB7_ground_truth.pdb")
top_pdb    = Path("data/docking/astemizole_top_pose.pdb")
truth_sdf  = Path("data/docking/8ZYO_XB7_ground_truth.sdf")
top_sdf    = Path("data/docking/astemizole_top_pose.sdf")

# Extract MODEL 1 of the docked PDBQT and convert to PDB
subprocess.run([".venv/bin/obabel", str(docked), "-O", str(top_pdb),
                "-f", "1", "-l", "1"], check=True, capture_output=True)

# PDB → SDF (heavy atoms only) for both
for src, dst in [(truth_pdb, truth_sdf), (top_pdb, top_sdf)]:
    subprocess.run([".venv/bin/obabel", str(src), "-O", str(dst), "-d"],
                   check=True, capture_output=True)

ref = next(Chem.SDMolSupplier(str(truth_sdf), removeHs=True, sanitize=True))
prb = next(Chem.SDMolSupplier(str(top_sdf),   removeHs=True, sanitize=True))
assert Chem.MolToSmiles(ref) == Chem.MolToSmiles(prb), "molecules differ — bond inference disagreed"

matches = ref.GetSubstructMatches(prb, uniquify=False, useChirality=False)
ref_conf = ref.GetConformer(); prb_conf = prb.GetConformer()
n = prb.GetNumAtoms()
rmsds = []
for m in matches:
    s = 0.0
    for i in range(n):
        p1 = prb_conf.GetAtomPosition(i); p2 = ref_conf.GetAtomPosition(m[i])
        s += (p1.x-p2.x)**2 + (p1.y-p2.y)**2 + (p1.z-p2.z)**2
    rmsds.append((s/n) ** 0.5)
best  = min(rmsds)

# Top Vina score from REMARK
top_score = next(float(l.split()[3]) for l in docked.read_text().splitlines()
                 if l.startswith("REMARK VINA RESULT"))

verdict = "PASS (≤ 2 Å)" if best <= 2.0 else (
          "BORDERLINE (≤ 2.5 Å)" if best <= 2.5 else "FAIL (> 2.5 Å)")
print(f"automorphisms tried : {len(matches)}")
print(f"best in-place RMSD  : {best:.3f} Å")
print(f"Vina top affinity   : {top_score:.3f} kcal/mol")
print(f"VALIDATION GATE     : {verdict}")


### Result — what this means

If the gate above prints **PASS**, the docking setup (receptor PDBQT,
box centre and size, ligand prep pipeline, Vina parameters) is *demonstrably
capable* of recovering an experimentally-known hERG–blocker pose. That gives
us the green light to dock our four poster compounds in Stage 4 with some
confidence that the *machinery* is sound — i.e. any failures there are due
to the compound, not the pipeline.

If the gate prints **FAIL**, the issue is in our setup (most likely the box
is wrongly placed, the receptor PDBQT has bad atom types, or the ligand prep
mis-typed a key atom). We would NOT proceed to dock unknowns in that case
because we'd have no way to interpret their scores.

> Note: passing this gate does **not** mean Vina will be accurate for our
> unknown compounds. Astemizole is exactly the molecule the cavity was
> co-resolved with — induced-fit effects are baked into the receptor. For
> compounds the receptor wasn't crystallised with, the same docking setup
> may overpack or under-rank. Re-docking is a *necessary* check, not a
> *sufficient* one.


## Stage 4 — Docking the poster compounds

The Stage 3 validation passed (RMSD 0.71 Å for re-docked astemizole), so the
pipeline is demonstrably capable of recovering a known pose in this cavity. Now
we apply it to our actual poster compounds:

- **Cisapride**, **Quinidine**, **Terfenadine** — known clinical hERG blockers
- **Rimantadine**, **Oleandomycin** — predicted non-blockers (Rimantadine is small;
  Oleandomycin is a macrolide much larger than astemizole's pocket)

Same receptor, same box, same Vina parameters as the validation run. SMILES come
directly from `data/worked_examples.csv` (the project's canonical source — same
SMILES that the chemistry model trained / predicted on, so docking results are
directly comparable to `p_pred` and `APD90`).

> ⚠️ **What these poses *are* and *aren't*:**
> - ✅ Vina-predicted poses against the experimentally-resolved hERG cavity.
> - ⚠️ The cavity in 8ZYO has been *induced-fit shaped around astemizole*. A drug
>   that's a different shape will not necessarily get the receptor it "deserves"
>   — Vina sees a static pocket. So treat affinities as **approximate and
>   comparative** (rank order, blocker vs non-blocker separation), not absolute
>   ΔG values.
> - ❌ Not validated against per-compound RMSD (no co-crystal exists for the
>   other 4). Only astemizole was structurally validated.


In [ ]:
# === 1) Dock 5 drugs (4 poster compounds + Rimantadine bonus) ===
import os, subprocess, time, json
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

DRUGS = ["OLEANDOMYCIN", "RIMANTADINE", "TERFENADINE", "QUINIDINE", "CISAPRIDE"]
we = pd.read_csv("data/worked_examples.csv").set_index("Drug_ID")
DOCK_DIR = Path("data/docking"); DOCK_DIR.mkdir(parents=True, exist_ok=True)
box = dict(l.split() for l in Path("data/docking/box_params.txt").read_text().splitlines())

results = []
for name in DRUGS:
    row = we.loc[name]
    sdf, pdbqt, docked, top_pdb = [
        DOCK_DIR / f"{name.lower()}_{s}" for s in
        ("query.sdf", "query.pdbqt", "docked.pdbqt", "top_pose.pdb")
    ]
    print(f"--- {name} ---")
    mol = Chem.MolFromSmiles(row.smiles); assert mol is not None
    mol_h = Chem.AddHs(mol)
    p = AllChem.ETKDGv3(); p.randomSeed = 42
    if AllChem.EmbedMolecule(mol_h, p) == -1:
        AllChem.EmbedMolecule(mol_h, useRandomCoords=True, randomSeed=42)
    try: AllChem.MMFFOptimizeMolecule(mol_h, maxIters=500)
    except Exception: pass
    Chem.SDWriter(str(sdf)).write(mol_h)

    r = subprocess.run([".venv/bin/mk_prepare_ligand.py", "-i", str(sdf), "-o", str(pdbqt)],
                       capture_output=True, text=True)
    assert r.returncode == 0 and pdbqt.exists(), f"meeko failed for {name}: {r.stderr[:300]}"
    rb = next((int(l.split()[1]) for l in pdbqt.read_text().splitlines()
               if l.startswith("TORSDOF")), None)

    t0 = time.time()
    r = subprocess.run([
        "tools/vina",
        "--receptor", "data/docking/8ZYO_receptor.pdbqt",
        "--ligand",   str(pdbqt),
        "--center_x", box["center_x"], "--center_y", box["center_y"], "--center_z", box["center_z"],
        "--size_x",   box["size_x"],   "--size_y",   box["size_y"],   "--size_z",   box["size_z"],
        "--exhaustiveness", "16", "--num_modes", "9", "--seed", "42",
        "--out", str(docked),
    ], capture_output=True, text=True)
    elapsed = time.time() - t0
    assert r.returncode == 0, f"Vina failed for {name}: {r.stderr[:300]}"
    score = next(float(l.split()[3]) for l in docked.read_text().splitlines()
                 if l.startswith("REMARK VINA RESULT"))
    subprocess.run([".venv/bin/obabel", str(docked), "-O", str(top_pdb),
                    "-f", "1", "-l", "1"], capture_output=True, check=False)
    results.append({
        "drug": name, "Y": int(row.Y), "p_pred": float(row.p_pred),
        "APD90": float(row.APD90), "qNet": float(row.qNet),
        "vina_score": float(score), "torsdof": rb,
        "n_heavy": mol.GetNumHeavyAtoms(), "elapsed_s": round(elapsed, 1),
        "top_pose_pdb": str(top_pdb),
    })
    print(f"  Vina={score:+.2f} kcal/mol  ({elapsed:.1f}s, {rb} rotatable bonds, {mol.GetNumHeavyAtoms()} heavy)")

Path("data/docking/stage4_results.json").write_text(json.dumps(results, indent=2))
print(f"\n✅ {len(results)}/{len(DRUGS)} drugs docked")


### 2) Results table: docking vs chemistry-model vs simulation

For each drug, we show the Vina top affinity alongside the project's predictions
(chemistry-model `p_pred`, O'Hara-Rudy `APD90`) and the ground truth label.


In [ ]:
# === 2) Build the results table + rank-correlation summary ===
import json, pandas as pd, numpy as np
from scipy.stats import pearsonr, spearmanr

results = json.load(open("data/docking/stage4_results.json"))
df = pd.DataFrame(results)
df["truth"]    = df["Y"].map({0: "non-blocker", 1: "blocker"})
df["outcome"]  = df["drug"].map({
    "OLEANDOMYCIN": "OK clinically (macrolide; no QT signal)",
    "RIMANTADINE":  "OK clinically (small antiviral; no QT signal)",
    "TERFENADINE":  "Withdrawn 1997 (QT prolongation)",
    "QUINIDINE":    "Antiarrhythmic — Class Ia; known QT risk",
    "CISAPRIDE":    "Withdrawn 2000 (QT prolongation, TdP)",
})
table = df[["drug", "truth", "vina_score", "p_pred", "APD90", "qNet", "outcome"]]\
    .rename(columns={"drug": "Drug", "vina_score": "Vina (kcal/mol)",
                     "p_pred": "p_pred", "APD90": "APD90 (ms)",
                     "qNet": "qNet (μC/μF)", "truth": "truth label",
                     "outcome": "clinical outcome"})\
    .sort_values("Vina (kcal/mol)").reset_index(drop=True)

print(table.to_string(index=False))

# Rank-order analysis: stronger binding (more negative Vina) should correlate
# with higher p_pred and higher APD90.
print("\n--- Agreement between Vina and the project's chemistry/simulation models ---")
v, p, a = df["vina_score"].values, df["p_pred"].values, df["APD90"].values
print(f"  Vina vs p_pred  :  pearson r = {pearsonr(v, p)[0]:+.3f}   spearman ρ = {spearmanr(v, p)[0]:+.3f}")
print(f"  Vina vs APD90   :  pearson r = {pearsonr(v, a)[0]:+.3f}   spearman ρ = {spearmanr(v, a)[0]:+.3f}")
print("\n(Negative correlation is the expected sign: lower Vina score = stronger predicted binder)")
table


### 3) Per-drug pose renders

For each drug we show **two things**:
1. A **static PNG** (`figures/dock_<drug>.png`) — Cα backbone trace (within 22 Å of
   the ligand) in light grey, ligand in CPK colours, two views (side + top
   down-the-pore). This is the file the poster build can pick up.
2. An **interactive py3Dmol cell** — grey cartoon protein + green-sticks ligand +
   translucent surface, zoomed to the pocket. Same idiom as the astemizole image
   in Part 4 of `hERG_Structure_Drugs.ipynb`.

The static PNG uses matplotlib + biopython rather than PyMOL because the
`pymol-open-source` wheel on PyPI for cp312-arm64 has hardcoded library paths
to the wheel-builder's mamba environment and won't load (and a from-source
PyMOL build is the rabbit hole we explicitly avoided in the feasibility report).
The matplotlib render is functional, not glamorous — for a final poster image
we'd export the py3Dmol view manually from a browser.


In [ ]:
# === 3a) Save a static PNG per drug ===
import json, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from Bio.PDB import PDBParser

CPK = {"C": "#444444", "N": "#2266ff", "O": "#dd2222", "F": "#33aa33",
       "CL": "#1f9a1f", "S": "#dddd22", "P": "#ee9933"}

def _parse_lig(pdb_path):
    atoms = []
    for line in Path(pdb_path).read_text().splitlines():
        if line.startswith(("ATOM", "HETATM")):
            elem = line[76:78].strip() or line[12:14].strip()
            if elem == "H": continue
            atoms.append((elem.upper(),
                          np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])))
    return atoms

def _bonds(atoms, max_d=1.85):
    c = np.array([x[1] for x in atoms]); n = len(atoms); bs = []
    for i in range(n):
        for j in range(i+1, n):
            if np.linalg.norm(c[i]-c[j]) <= max_d: bs.append((i, j))
    return bs

def _bb(near, R=22.0):
    parser = PDBParser(QUIET=True)
    s = parser.get_structure("rec", "data/docking/8ZYO_protein.pdb")
    chains = {}
    for model in s:
        for chain in model:
            for res in chain:
                if "CA" in res:
                    c = res["CA"].get_coord()
                    if np.linalg.norm(c - near) < R:
                        chains.setdefault(chain.id, []).append((res.id[1], c))
    for cid in chains: chains[cid].sort(key=lambda t: t[0])
    return chains

def render(drug, vina, ligand_pdb, out_png):
    atoms = _parse_lig(ligand_pdb)
    coords = np.array([c for _, c in atoms])
    centroid = coords.mean(axis=0)
    bonds = _bonds(atoms); bb = _bb(centroid)
    fig = plt.figure(figsize=(11, 5.5))
    for col, (az, el, lbl) in enumerate([(40, 12, "side view"),
                                          (-90, 88, "top view (down the pore)")]):
        ax = fig.add_subplot(1, 2, col+1, projection='3d')
        for cid, items in bb.items():
            xs, ys, zs = zip(*[(c[0], c[1], c[2]) for _, c in items])
            ax.plot(xs, ys, zs, color="#a0a0a0", lw=0.9, alpha=0.55)
            ax.scatter(xs, ys, zs, s=2, color="#a0a0a0", alpha=0.4)
        for i, j in bonds:
            ax.plot([coords[i,0], coords[j,0]],
                    [coords[i,1], coords[j,1]],
                    [coords[i,2], coords[j,2]],
                    color="#222", lw=1.4, alpha=0.85)
        ax.scatter(coords[:,0], coords[:,1], coords[:,2],
                   c=[CPK.get(e, "magenta") for e, _ in atoms],
                   s=90, edgecolors="black", linewidths=0.6)
        R = 14
        ax.set_xlim(centroid[0]-R, centroid[0]+R)
        ax.set_ylim(centroid[1]-R, centroid[1]+R)
        ax.set_zlim(centroid[2]-R, centroid[2]+R)
        ax.view_init(elev=el, azim=az)
        ax.set_title(lbl, fontsize=10)
        ax.set_xlabel("X (Å)", fontsize=8); ax.set_ylabel("Y (Å)", fontsize=8); ax.set_zlabel("Z (Å)", fontsize=8)
        ax.tick_params(labelsize=7)
    fig.suptitle(f"{drug.title()} — docked into hERG pore (Vina = {vina:+.2f} kcal/mol)",
                 fontsize=13, y=0.99)
    fig.tight_layout()
    fig.savefig(out_png, dpi=140, facecolor="white")
    plt.close(fig)

Path("figures").mkdir(exist_ok=True)
results = json.load(open("data/docking/stage4_results.json"))
for r in results:
    out = Path(f"figures/dock_{r['drug'].lower()}.png")
    render(r['drug'], r['vina_score'], r['top_pose_pdb'], out)
    print(f"  saved {out}  ({out.stat().st_size:,} B)")


In [ ]:
# === 3b) Interactive py3Dmol view per drug ===
# Renders all 5 in a grid of HTML viewers. Same style as the astemizole image
# in Part 4 of hERG_Structure_Drugs.ipynb (grey cartoon + green stick ligand +
# translucent surface). Drag to rotate; scroll to zoom.
import json, py3Dmol
from pathlib import Path
from IPython.display import display, HTML

RECEPTOR = Path("data/docking/8ZYO_protein.pdb").read_text()
results = json.load(open("data/docking/stage4_results.json"))

for r in results:
    lig = Path(r["top_pose_pdb"]).read_text()
    v = py3Dmol.view(width=520, height=400)
    v.addModel(RECEPTOR, "pdb")
    v.setStyle({"hetflag": False},
               {"cartoon": {"color": "lightgrey", "opacity": 0.9}})
    v.addModel(lig, "pdb")
    v.setStyle({"model": 1},
               {"stick": {"colorscheme": "greenCarbon", "radius": 0.22}})
    v.addSurface(py3Dmol.SAS,
                 {"opacity": 0.45, "color": "lightgreen"},
                 {"model": 1})
    v.setBackgroundColor("white")
    v.zoomTo({"model": 1}); v.zoom(0.85)
    display(HTML(f"<h4 style='margin-bottom:4px'>{r['drug'].title()} — Vina {r['vina_score']:+.2f} kcal/mol "
                 f"(predicted p={r['p_pred']:.3f}, APD90={r['APD90']:.0f} ms)</h4>"))
    v.show()


### 4) Honest interpretation

**What does the docking actually tell us?**

1. **Blocker / non-blocker separation is clean.** The three known clinical blockers
   (Terfenadine, Quinidine, Cisapride) all score between **-8.6 and -9.8 kcal/mol**;
   the two non-blockers score worse — Rimantadine **-6.8** (small molecule, doesn't
   fill the cavity well) and Oleandomycin **-3.0** (a macrolide far too large for
   the cavity to wrap around). The astemizole positive control scored -12.4. So
   the structural argument **agrees with the chemistry-model classification**.

2. **Rank-order within the blockers disagrees.** Vina ranks
   Terfenadine > Quinidine > Cisapride (most negative first); the chemistry model
   says Cisapride > Quinidine > Terfenadine; the simulation's APD90 puts
   Cisapride first too. The Spearman correlation between Vina and `p_pred` across
   all five drugs is **ρ ≈ -0.60** — moderate, not perfect. Two reasonable
   reasons for the disagreement:
   - **Induced-fit bias.** 8ZYO's cavity was crystallographically shaped around
     astemizole; Vina sees a frozen pocket. The chemistry model isn't constrained
     by any single co-crystal conformation.
   - **Vina's scoring is a coarse free-energy approximation.** The chemistry
     model has learned non-additive features from thousands of measured Kᵢ
     values — Vina hasn't.

3. **Oleandomycin's -3.0 score is the most informative result.** A macrolide of
   48 heavy atoms simply does not fit in a cavity scaled around the 34-heavy-atom
   astemizole. That's a *structural* argument for why it's a non-blocker — Vina
   couldn't find a tight pose. This is exactly the kind of explanation a
   chemistry-model probability *can't* give you on its own.

**Bottom line:** docking is a useful **second opinion** here. It confirms the
binary call (blocker vs not), and it adds a structural narrative for the
non-blocker case. It is **not a substitute** for the chemistry model — its
quantitative ranking inside the blocker class is less reliable than `p_pred`,
and that's the expected behaviour given the induced-fit caveat.
